In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [4]:
prompt = ChatPromptTemplate.from_template("tell me a curious fact about {soccer_player}")

output_parser = StrOutputParser()

In [5]:
chain = prompt | model | output_parser

chain.invoke({"soccer_player": "Ronaldo"})

'One curious fact about Cristiano Ronaldo is that he is known for his incredible work ethic and dedication to his fitness. He reportedly has the physical capacity of a 20-year-old, despite being in his mid-30s, thanks to his intense training and strict diet. Ronaldo is known to have a meticulous approach to his fitness, constantly monitoring his body fat percentage and muscle mass to ensure he is in peak condition.'

## Use of .bind() to add arguments to a Runnable in a LCEL Chain
* For example, we can add an argument to stop the model response when it reaches the word "Ronaldo":

In [7]:
chain = prompt | model.bind(stop=["Ronaldo"]) | output_parser
chain.invoke({"soccer_player": "Ronaldo"})

'One curious fact about '

# Combining LCEL Chains

In [8]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo")

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("tell me a sentence about {politician}")

chain = prompt | model | StrOutputParser()

In [10]:
chain.invoke("Chamberlain")

'Chamberlain was a British Prime Minister known for his policy of appeasement towards Nazi Germany before World War II.'

# Combined chain

In [11]:
historian_prompt = ChatPromptTemplate.from_template("Was {politician} positive for Humanity?")

composed_chain = {"politician": chain} | historian_prompt | model | StrOutputParser()

In [12]:
composed_chain.invoke({"politician": "Lincoln"})

"Yes, Abraham Lincoln's leadership during the Civil War and his dedication to abolishing slavery had a positive impact on humanity. His Emancipation Proclamation and his push for the passage of the 13th Amendment helped to end the institution of slavery in the United States, leading to greater freedom and equality for all Americans. Lincoln's commitment to preserving the union and his belief in the principles of democracy continue to inspire people around the world."

## Another example: a chain inside another chain

In [14]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt1 = ChatPromptTemplate.from_template("what is the country {politician} is from?")
prompt2 = ChatPromptTemplate.from_template(
    "what continent is the country {country} in? respond in {language}"
)

model = ChatOpenAI()

chain1 = prompt1 | model | StrOutputParser()

chain2 = (
    {"country": chain1, "language": itemgetter("language")}
    | prompt2
    | model
    | StrOutputParser()
)

chain2.invoke({"politician": "Miterrand", "language": "Telugu"})

'ఫ్రాన్స్ దేశం యొక్క రాయిరెడ్రాండ్ ఎక్కువతో యూరప్ యిన ఉత్తరామేరిక భక్తివెలుపు భాగాలో ఉంది.'

# LCEL chain at work in a typical RAG app

In [15]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [22]:
import bs4
from langsmith import Client as LangSmithClient  # Use native LangSmith client
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

retriever = vectorstore.as_retriever()

# Pull the standard RAG prompt securely
client = LangSmithClient()
prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [23]:
rag_chain.invoke("What is Task Decomposition?")

'Task decomposition is a technique where complex tasks are broken down into smaller and simpler steps. This allows for a systematic approach to solving the task and helps in enhancing model performance on complex tasks. Techniques like Chain of Thought and Tree of Thoughts are examples of methods used for task decomposition.'